# M2.S3 — Distributed-memory computing with MPI
## Interactive HPC notebook

This notebook accompanies **M2.S3 — Distributed-memory computing with MPI**.

The key transition from the previous session is:

> **OpenMP shares memory. MPI moves data.**

We will make the main lecture concepts visible by running small MPI programs:

1. identify processes with **rank, size and communicator**;
2. see that every process owns **separate memory**;
3. exchange data with **send / receive / tags**;
4. reason about **message ordering and deadlock**;
5. use **broadcast, scatter, gather and reduce**;
6. observe that **communication is not free**;
7. connect **Slurm tasks** to MPI ranks across nodes.

### Classroom method
> **PREDICT → RUN → OBSERVE → EXPLAIN**

### Recommended environment
- IE/SciTech JupyterHub
- Python 3 kernel
- MPI compiler/runtime (`mpicc`, `mpirun`/`mpiexec`)
- Slurm for the multi-node activity

The notebook does **not** install MPI or system packages. Cluster-dependent activities fail gracefully.

# 0 — Environment check

### Predict
Before running the cell:

- Is `mpicc` available?
- Is `mpirun` or `mpiexec` available?
- Is Slurm visible from this Jupyter environment?
- Does seeing many CPUs mean that you automatically own them?

In [ ]:
import os
import platform
import shutil
import subprocess
import textwrap
import time

print('Host:', platform.node())
print('User:', os.environ.get('USER', 'unknown'))
print('Visible CPUs:', os.cpu_count())
print('mpicc:', shutil.which('mpicc'))
print('mpirun:', shutil.which('mpirun'))
print('mpiexec:', shutil.which('mpiexec'))
print('srun:', shutil.which('srun'))
print('sinfo:', shutil.which('sinfo'))

if shutil.which('mpicc'):
    p = subprocess.run(['mpicc', '--version'], capture_output=True, text=True)
    print('\nMPI compiler wrapper:')
    print((p.stdout or p.stderr).splitlines()[0] if (p.stdout or p.stderr) else 'available')
else:
    print('\nmpicc is not available. Run the MPI activities on the SciTech HPC environment.')

In [ ]:
def compile_mpi(source_file, exe_file):
    if shutil.which('mpicc') is None:
        print('mpicc not found — run this activity on the SciTech HPC environment.')
        return False
    p = subprocess.run(['mpicc', '-O2', source_file, '-o', exe_file],
                       capture_output=True, text=True)
    if p.returncode != 0:
        print('Compilation failed:')
        print(p.stderr)
        return False
    return True

def run_mpi(exe_file, n=4, timeout=30):
    runner = shutil.which('mpirun') or shutil.which('mpiexec')
    if runner is None:
        print('mpirun/mpiexec not found — use the Slurm activity later in the notebook.')
        return None
    env = os.environ.copy()
    # Harmless on normal accounts; useful in some containerized teaching environments.
    env.setdefault('OMPI_ALLOW_RUN_AS_ROOT', '1')
    env.setdefault('OMPI_ALLOW_RUN_AS_ROOT_CONFIRM', '1')
    cmd = [runner, '-np', str(n), './' + exe_file]
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=timeout)
    except subprocess.TimeoutExpired:
        print('MPI run timed out. Do not keep retrying: use the SciTech/Slurm execution path.')
        return None
    if p.stdout:
        print(p.stdout, end='')
    if p.returncode != 0:
        print('\nMPI runtime returned an error:')
        print(p.stderr)
        print('If local MPI launching is restricted, use the Slurm section in this notebook.')
    return p

### Explain
The MPI runtime launches **processes**. Slurm, when used, allocates resources to your job.

Those are related but different decisions.

# 1 — Rank, size and communicator
### Slide connection: every MPI process needs an identity

Every process executes the same program, but each process can discover:

- its **rank**: identity inside a communicator;
- the **size**: number of processes in that communicator;
- the processor/host where it is running.

### Predict
If we launch 4 processes:

1. What ranks should exist?
2. Will rank 3 have higher priority than rank 1?
3. Will the printed lines necessarily appear in rank order?

In [ ]:
hello_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank, size, len;
    char name[MPI_MAX_PROCESSOR_NAME];
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    MPI_Get_processor_name(name, &len);

    printf("Hello from rank %d of %d on %s\n", rank, size, name);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_hello.c', 'w') as f:
    f.write(hello_src)

if compile_mpi('mpi_hello.c', 'mpi_hello'):
    run_mpi('mpi_hello', 4)

### Observe
- Ranks should be `0 ... size-1`.
- Rank is an **identifier**, not a resource priority.
- Output ordering can vary because processes execute asynchronously.

### Try it
Change the launch from 4 processes to 2, then 6 (if your environment permits it).

### Explain
Why can the same executable behave differently on each process even though every process starts from the same source code?

# 2 — Separate memory: changing Rank 0 does not change Rank 1
### Slide connection: MPI does not make remote memory shared

Each process has its own local memory.

### Predict
Each rank starts with `value = rank * 10`.

Rank 0 then changes **its own** value to 999.

Will any other rank automatically see 999?

In [ ]:
memory_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int value = rank * 10;
    printf("Rank %d initially has value=%d\n", rank, value);

    MPI_Barrier(MPI_COMM_WORLD);
    if (rank == 0) value = 999;
    MPI_Barrier(MPI_COMM_WORLD);

    printf("Rank %d after Rank 0 changes its copy: value=%d\n", rank, value);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_separate_memory.c', 'w') as f:
    f.write(memory_src)

if compile_mpi('mpi_separate_memory.c', 'mpi_separate_memory'):
    run_mpi('mpi_separate_memory', 4)

### Observe
Only Rank 0 changes to 999.

### Explain
How is this fundamentally different from OpenMP threads reading and writing one shared variable?

# 3 — Point-to-point communication: send, receive and tag
### Slide connection: one sender · one receiver · one matching communication

We model a weather-boundary update:

- Rank 0 owns a local boundary temperature;
- Rank 1 cannot read Rank 0's variable directly;
- Rank 0 sends a **copy**;
- Rank 1 receives it using matching source/tag/communicator information.

### Predict
What should Rank 1 know after the receive?

Can Rank 1 discover which rank sent the message and which tag was used?

In [ ]:
p2p_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 2) {
        if (rank == 0) printf("Run this example with exactly 2 processes.\n");
        MPI_Finalize();
        return 0;
    }

    const int tag = 42;
    if (rank == 0) {
        double boundary_temp = 18.4;
        printf("Rank 0 local value before send = %.1f C\n", boundary_temp);
        MPI_Send(&boundary_temp, 1, MPI_DOUBLE, 1, tag, MPI_COMM_WORLD);
    } else {
        double received_temp = -999.0;
        MPI_Status status;
        MPI_Recv(&received_temp, 1, MPI_DOUBLE, 0, tag, MPI_COMM_WORLD, &status);
        printf("Rank 1 received %.1f C from rank %d with tag %d\n",
               received_temp, status.MPI_SOURCE, status.MPI_TAG);
    }

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_point_to_point.c', 'w') as f:
    f.write(p2p_src)

if compile_mpi('mpi_point_to_point.c', 'mpi_point_to_point'):
    run_mpi('mpi_point_to_point', 2)

### Observe
The receiver obtains a copy through explicit communication.

### Explain
A message match depends on **source/destination + tag + communicator**. Why is the tag useful when the same pair of ranks exchanges several different kinds of data?

# 4 — Message ordering and deadlock
### Slide connection: correct data · wrong communication order

Consider this unsafe pattern with two ranks:

```c
// Rank 0
MPI_Recv(... from Rank 1 ...);
MPI_Send(... to Rank 1 ...);

// Rank 1
MPI_Recv(... from Rank 0 ...);
MPI_Send(... to Rank 0 ...);
```

### Predict
What event can make progress first?

**Do not execute a deliberately hanging program in a shared class notebook.**

Instead, run the repaired version below: one side sends first, the other receives first.

In [ ]:
ordered_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 2) {
        if (rank == 0) printf("Run this example with exactly 2 processes.\n");
        MPI_Finalize();
        return 0;
    }

    int send_value = (rank == 0) ? 100 : 200;
    int recv_value = -1;

    if (rank == 0) {
        MPI_Send(&send_value, 1, MPI_INT, 1, 7, MPI_COMM_WORLD);
        MPI_Recv(&recv_value, 1, MPI_INT, 1, 8, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    } else {
        MPI_Recv(&recv_value, 1, MPI_INT, 0, 7, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Send(&send_value, 1, MPI_INT, 0, 8, MPI_COMM_WORLD);
    }

    printf("Rank %d received %d\n", rank, recv_value);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_ordered_exchange.c', 'w') as f:
    f.write(ordered_src)

if compile_mpi('mpi_ordered_exchange.c', 'mpi_ordered_exchange'):
    run_mpi('mpi_ordered_exchange', 2)

### Observe
The first matching send/receive can complete, so the second exchange can then proceed.

### Explain
- Why did the original receive-first/receive-first pattern have no possible first step?
- How could non-blocking communication (`MPI_Isend`, `MPI_Irecv`, `MPI_Wait`) provide another solution?

# 5 — Collectives: match the operation to the communication pattern
### Slide connection: broadcast · scatter · gather · reduce

We will execute four common patterns in one program:

1. **Broadcast** — Rank 0 shares a timestep with everyone.
2. **Scatter** — Rank 0 divides an 8-element image among 4 ranks.
3. **Gather** — processed pieces return to Rank 0.
4. **Reduce** — local energies become one total energy.

### Predict
With 4 processes:

- What timestep should every rank print?
- Which two image values should each rank receive?
- What should the gathered processed image contain?
- If local energy is `rank + 1`, what should the reduced total be?

In [ ]:
collective_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 4) {
        if (rank == 0) printf("Run this example with exactly 4 processes.\n");
        MPI_Finalize();
        return 0;
    }

    // 1) Broadcast
    int timestep = (rank == 0) ? 60 : -1;
    MPI_Bcast(&timestep, 1, MPI_INT, 0, MPI_COMM_WORLD);
    printf("Rank %d: timestep=%d\n", rank, timestep);

    MPI_Barrier(MPI_COMM_WORLD);

    // 2) Scatter
    int image[8] = {1,2,3,4,5,6,7,8};
    int chunk[2] = {-1,-1};
    MPI_Scatter(image, 2, MPI_INT, chunk, 2, MPI_INT, 0, MPI_COMM_WORLD);
    printf("Rank %d received image chunk [%d,%d]\n", rank, chunk[0], chunk[1]);

    // Local processing
    chunk[0] *= 10;
    chunk[1] *= 10;

    // 3) Gather
    int processed[8] = {0};
    MPI_Gather(chunk, 2, MPI_INT, processed, 2, MPI_INT, 0, MPI_COMM_WORLD);
    if (rank == 0) {
        printf("Gathered processed image:");
        for (int i = 0; i < 8; ++i) printf(" %d", processed[i]);
        printf("\n");
    }

    // 4) Reduce
    int local_energy = rank + 1;
    int total_energy = 0;
    MPI_Reduce(&local_energy, &total_energy, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);
    if (rank == 0)
        printf("Reduced total energy = %d\n", total_energy);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_collectives.c', 'w') as f:
    f.write(collective_src)

if compile_mpi('mpi_collectives.c', 'mpi_collectives'):
    run_mpi('mpi_collectives', 4)

### Observe
- **Broadcast:** one value → everyone.
- **Scatter:** different data pieces → different ranks.
- **Gather:** different pieces → one root.
- **Reduce:** local values → one combined result.

### Explain
Could you reproduce all of these patterns using many individual sends and receives?

Why is using the collective that matches the problem usually clearer?

# 6 — Communication is not free
### Slide connection: moving data and waiting can limit speedup

This optional micro-experiment measures a simple ping-pong between two MPI ranks.

It does **not** represent a full cluster benchmark, especially if both ranks run on one node. Its purpose is to make one point visible:

> Sending larger messages and performing more communication costs time.

### Predict
Which message size should have the smallest absolute round-trip time?

Why might larger messages achieve higher effective bandwidth even though each round trip takes longer?

In [ ]:
ping_src = r'''
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

static void test_size(int count, int reps, int rank) {
    double *buf = (double*)malloc((size_t)count * sizeof(double));
    for (int i = 0; i < count; ++i) buf[i] = (double)i;

    MPI_Barrier(MPI_COMM_WORLD);
    double t0 = MPI_Wtime();

    for (int r = 0; r < reps; ++r) {
        if (rank == 0) {
            MPI_Send(buf, count, MPI_DOUBLE, 1, 1, MPI_COMM_WORLD);
            MPI_Recv(buf, count, MPI_DOUBLE, 1, 2, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        } else {
            MPI_Recv(buf, count, MPI_DOUBLE, 0, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            MPI_Send(buf, count, MPI_DOUBLE, 0, 2, MPI_COMM_WORLD);
        }
    }

    double elapsed = MPI_Wtime() - t0;
    if (rank == 0) {
        double bytes = (double)count * sizeof(double);
        double avg_rtt_us = elapsed * 1e6 / reps;
        double mb_s = (2.0 * bytes * reps) / elapsed / 1e6;
        printf("%9.1f KiB : avg round trip = %9.2f us, effective transfer = %9.1f MB/s\n",
               bytes / 1024.0, avg_rtt_us, mb_s);
    }
    free(buf);
}

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 2) {
        if (rank == 0) printf("Run this example with exactly 2 processes.\n");
        MPI_Finalize();
        return 0;
    }

    test_size(1,       200, rank);      // 8 bytes
    test_size(1024,    100, rank);      // 8 KiB
    test_size(131072,   20, rank);      // 1 MiB

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_pingpong_cost.c', 'w') as f:
    f.write(ping_src)

if compile_mpi('mpi_pingpong_cost.c', 'mpi_pingpong_cost'):
    run_mpi('mpi_pingpong_cost', 2, timeout=45)

### Observe
Interpret the result qualitatively, not as a definitive cluster benchmark.

### Explain
If a simulation keeps splitting the problem into smaller regions, why can communication and waiting eventually dominate useful computation?

# 7 — Slurm allocation → MPI ranks
### Slide connection: scheduler allocation and MPI process identity

The lecture example requests:

```text
2 nodes
4 MPI tasks
2 tasks per node
```

The batch script below is created but **not submitted automatically**.

### Predict
If Slurm grants the request, how many rank IDs should the program print?

Could two ranks share the same hostname?

In [ ]:
slurm_script = '''#!/bin/bash
#SBATCH --job-name=m2s3_mpi
#SBATCH --nodes=2
#SBATCH --ntasks=4
#SBATCH --ntasks-per-node=2
#SBATCH --time=00:02:00
#SBATCH --output=m2s3_mpi-%j.out

echo "Allocated nodes:"
scontrol show hostnames "$SLURM_JOB_NODELIST"
echo

mpicc -O2 mpi_hello.c -o mpi_hello
srun ./mpi_hello
'''

with open('m2s3_mpi.slurm', 'w') as f:
    f.write(slurm_script)

print(slurm_script)

In [ ]:
if shutil.which('sinfo') is None:
    print('Slurm is not visible here. Submit m2s3_mpi.slurm from the SciTech cluster environment.')
else:
    print('Slurm is available.')
    print('Inspect the cluster first with: sinfo')
    print('Then, if your account permits multi-node jobs: sbatch m2s3_mpi.slurm')
    print('Track it with: squeue -u $USER')

### Important
Do not invent a partition or account name. If SciTech requires `--partition` or `--account`, use the values provided by SciTech.

### Explain
- **Slurm** allocates and places resources.
- **MPI** gives each launched process a rank and coordinates communication.

Why is an MPI rank not the same thing as a physical CPU core or node?

# Challenge — Distributed image processing

Work in pairs.

You have a grayscale image represented as 16 integer pixels on Rank 0:

```text
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
```

Your task is to design an MPI solution for 4 ranks:

1. Rank 0 owns the full image initially.
2. Split it evenly across 4 ranks.
3. Each rank applies `pixel = 255 - pixel` to its local chunk.
4. Reconstruct the final image on Rank 0.
5. Each rank computes the sum of its processed pixels.
6. Rank 0 obtains the global sum.

### Before coding, choose the MPI operation for each step
- distribute image chunks → **?**
- reconstruct image → **?**
- combine local sums → **?**

Use the collective example above as your starting point rather than typing a large program from scratch.

# What did we learn?

1. **MPI processes own separate memory.** Remote variables are not directly shared.
2. **Rank identifies a process inside a communicator.** Rank is not a priority.
3. **Point-to-point messages must match.** Source/destination, tag and communicator matter.
4. **Communication ordering matters.** A valid calculation can still hang if no process can make progress.
5. **Collectives express common group patterns:** broadcast, scatter, gather and reduce.
6. **Communication has a cost.** Good MPI designs do enough useful local work between communications.
7. **Slurm allocates resources; MPI processes run on those resources.**

## Next: M2.S4 — Accelerator computing

Distributed CPU processes are only one level of modern HPC. In the next session we ask:

> If every node also has a GPU, which work should stay on the CPU and which work should run on the accelerator?